In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langchain_groq import ChatGroq
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
llm=ChatGroq(model="openai/gpt-oss-20b",temperature=0)

In [10]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explanation:str

In [11]:
def generate_joke(state:JokeState):
    prompt=f'generate a joke on the topic {state["topic"]}'
    response=llm.invoke(prompt).content
    return {'joke':response}



In [12]:
def generate_explanation(state:JokeState):
    prompt=f'write an explanation for the joke {state["joke"]}'
    response=llm.invoke(prompt).content
    return {'explanation':response}

In [13]:
graph=StateGraph(JokeState)

graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

checkpointer=InMemorySaver()
workflow=graph.compile(checkpointer=checkpointer)

In [18]:
config1={"configurable":{"thread_id":"1"}}
workflow.invoke({'topic':'pasta'},config=config1)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti start a band?\n\nBecause it wanted to be a *fettuccine* rockstar—every time it hit a note, it *pasta* the beat!',
 'explanation': '**Why the joke works – a quick breakdown**\n\n| Element | What it is | Why it’s funny |\n|---------|------------|----------------|\n| **Spaghetti starts a band** | An absurd image – a noodle, not a person, forming a musical group. | The sheer silliness of a pasta “band” sets the stage for a pun. |\n| **“fettuccine rockstar”** | A play on the word *fettuccine* (a type of pasta) and the phrase *rockstar*. | It sounds like “fettuccine” is a *rockstar* in its own right, turning a food item into a celebrity. |\n| **“hit a note”** | A common musical phrase meaning to play a pitch. | It also hints at the noodle “hitting” a musical note, reinforcing the music‑pasta mash‑up. |\n| **“pasta the beat”** | A pun on *pasta* (the food) and the phrase *pass the beat* (or *past the beat*). | The joke replaces a normal music

In [19]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti start a band?\n\nBecause it wanted to be a *fettuccine* rockstar—every time it hit a note, it *pasta* the beat!', 'explanation': '**Why the joke works – a quick breakdown**\n\n| Element | What it is | Why it’s funny |\n|---------|------------|----------------|\n| **Spaghetti starts a band** | An absurd image – a noodle, not a person, forming a musical group. | The sheer silliness of a pasta “band” sets the stage for a pun. |\n| **“fettuccine rockstar”** | A play on the word *fettuccine* (a type of pasta) and the phrase *rockstar*. | It sounds like “fettuccine” is a *rockstar* in its own right, turning a food item into a celebrity. |\n| **“hit a note”** | A common musical phrase meaning to play a pitch. | It also hints at the noodle “hitting” a musical note, reinforcing the music‑pasta mash‑up. |\n| **“pasta the beat”** | A pun on *pasta* (the food) and the phrase *pass the beat* (or *past the beat*). | The joke repl

In [20]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti start a band?\n\nBecause it wanted to be a *fettuccine* rockstar—every time it hit a note, it *pasta* the beat!', 'explanation': '**Why the joke works – a quick breakdown**\n\n| Element | What it is | Why it’s funny |\n|---------|------------|----------------|\n| **Spaghetti starts a band** | An absurd image – a noodle, not a person, forming a musical group. | The sheer silliness of a pasta “band” sets the stage for a pun. |\n| **“fettuccine rockstar”** | A play on the word *fettuccine* (a type of pasta) and the phrase *rockstar*. | It sounds like “fettuccine” is a *rockstar* in its own right, turning a food item into a celebrity. |\n| **“hit a note”** | A common musical phrase meaning to play a pitch. | It also hints at the noodle “hitting” a musical note, reinforcing the music‑pasta mash‑up. |\n| **“pasta the beat”** | A pun on *pasta* (the food) and the phrase *pass the beat* (or *past the beat*). | The joke rep